<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>

# 2. RT-CV: Detection & Tracking

RT-CV (detection and tracking) finds the vehicles and people in every frame and follows each one over time. In
this notebook you add a traffic clip to VIOS and follow what RT-CV makes of it, from single boxes to tracklets to
the vehicles that stand still.

In this notebook you will:

- add a video to VIOS (video storage) and learn the three names that services use for it
- read RT-CV's detections from Kafka (the message bus) and Elasticsearch (the search index)
- draw detections on a frame and follow one vehicle's tracklet
- find the vehicles that stand still, from RT-CV's own tracklets
- search for objects by appearance with RT-CV's object embeddings, from an example object or from text

**Time:** 40 minutes. Section 2.13 is optional.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Part 1** &middot; about 40 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> Previous: [1. Inspect VSS Deployment](01-inspect-vss-deployment.ipynb) &middot; Next: [3. RT-VLM: Captioning & Prompting](03-rt-vlm-captioning-and-prompting.ipynb)


<img src="../images/course/map-ingest.png" width="960" alt="The VSS pipeline map with this notebook's path lit: your notebook, the VSS agent service and VIOS for the upload, then RT-CV, Kafka and Elasticsearch for the detections. RT-VLM is dimmed.">

In [ ]:
import json, time, requests
from pathlib import Path

import labtools
from course_setup import (BASE, upload_url, find_sensor, wait_for_detections,
                          clip_bytes, iso_offset, offset_seconds, es)
from cleanup import remember          # records what Part 1 creates, so the course cleanup can remove it

CLIP = "/dli/task/data/traffic_short.mp4"
NAME = "traffic_short.mp4"            # the filename you upload
SOURCE = Path(NAME).stem              # the file stem, traffic_short: RT-CV's name for this source
ANCHOR = "2025-01-01T00:00:00.000Z"   # the start time you give the upload
INDEX = "mdx-raw-2025-01-01"          # RT-CV's frame records for uploaded files

## 2.1 Inspect the starting state

List the videos VIOS holds and the indices in Elasticsearch. VIOS already lists the four search videos the course
added when the lab started, and Elasticsearch already holds their records. Notebook 4 searches them.

In [ ]:
r = requests.get(BASE + "/vios/vst/api/v1/sensor/list", timeout=20)
r.raise_for_status()
r.json()

In [ ]:
r = requests.get(BASE + "/es/_cat/indices/mdx-*,default_*?v", timeout=20)
r.raise_for_status()
print(r.text or "(no indices)")

## 2.2 Add a video to VIOS

Adding a video takes three calls. You ask the VSS agent service for an upload URL, send the file to VIOS, and tell
the VSS agent service that the upload is complete. The last call starts RT-CV on the video.

<img src="../images/course/ingest-handshake.gif" width="960" alt="A sequence diagram of the three-call upload: your notebook asks the VSS agent service (vss-agent) for an upload URL, rewrites its port from :80 to :30888, posts traffic_short.mp4 to VIOS with the start time 2025-01-01T00:00:00 and keeps the returned sensorId. It then calls /complete, which starts RT-CV, and reads the 16-second timeline back from VIOS.">

**1. Request an upload URL.** The VSS agent service returns the address to send the file to.

In [ ]:
r = requests.post(BASE + "/agent/api/v1/videos", json={"filename": NAME}, timeout=60)
r.raise_for_status()
agent_url = r.json()["url"]
agent_url

The URL carries the deployment's public port, 80. From inside the lab you reach VIOS on its internal port, 30888,
so `course_setup.upload_url()` swaps `:80/` for `:30888/`.

In [ ]:
target = upload_url(agent_url)
target

**2. Upload the file.** The upload API takes a file in chunks. This clip is small, so you send it as one chunk, and
the `nvstreamer-*` headers say so. The `metadata` part sets the recording's start time on the VIOS
timeline, `2025-01-01T00:00:00`. You use that start time again whenever you cut a clip from VIOS.

If VIOS already holds the clip from an earlier run, the cell reuses it. Set `REUPLOAD = True` only if you want a
second copy. Its records would also land under `traffic_short`, so the counts in this notebook would include both
copies.

In [ ]:
REUPLOAD = False   # True adds a second copy of the clip, even if VIOS already holds one

existing = find_sensor(SOURCE)   # the VIOS record named traffic_short or traffic_short_<n>, or None
reused = bool(existing) and not REUPLOAD
if reused:
    SENSOR_ID = existing["sensorId"]
    print("VIOS already holds this clip as", existing["name"], "so this run reuses it")
else:
    with open(CLIP, "rb") as fh:
        r = requests.post(
            target,
            headers={
                "nvstreamer-chunk-number":  "1",
                "nvstreamer-total-chunks":  "1",
                "nvstreamer-is-last-chunk": "true",
                "nvstreamer-identifier":    NAME,
                "nvstreamer-file-name":     NAME,
            },
            files={
                "metadata": (None, json.dumps({"timestamp": "2025-01-01T00:00:00"})),
                "file": (NAME, fh, "video/mp4"),
            },
            timeout=300,
        )
    r.raise_for_status()
    SENSOR_ID = r.json()["sensorId"]
    remember("video", SENSOR_ID, name=SOURCE)   # adds it to /dli/task/data/part1_resources.json
print("sensorId:", SENSOR_ID)

The cell saves the video's VIOS sensor id (`sensorId`) in `SENSOR_ID`. You cut clips with it. For a new upload, the cell also adds the video to `/dli/task/data/part1_resources.json`, the
list of what Part 1 created, so the course cleanup can remove it.

**3. Complete the upload.** This call starts the analysis: RT-CV (TrafficCamNet RT-DETR) takes the file. It
returns within about 30 seconds. For a reused clip
that RT-CV has already processed, the cell skips the call.

In [ ]:
# A reused clip was completed on an earlier run, unless that run stopped before this cell.
already = reused and es("POST", "/mdx-raw-*/_count",
                        json={"query": {"term": {"sensorId.keyword": SOURCE}}})["count"] > 0
if already:
    print("skipped: RT-CV already has records for the reused clip")
else:
    r = requests.post(BASE + "/agent/api/v1/videos/" + SENSOR_ID + "/complete",
                      json={"filename": NAME}, timeout=600)
    r.raise_for_status()
    print(r.json())

## 2.3 Look up the source and its timeline

One video now has three names. Use the right one for each service:

| Name | Value here | Use it for |
|---|---|---|
| File stem | `traffic_short` | RT-CV records (`sensorId` in `mdx-raw-*`) and Behavior Analytics tracklet summaries (`sensor.id` in `mdx-behavior-*`) |
| VIOS sensor id | the UUID in `SENSOR_ID` | cutting clips |
| VIOS name | `traffic_short_1` | display, and lookup with `find_sensor()`. VIOS adds the suffix, and no index uses this name |

RT-CV takes the stem from the filename you send to `/complete`. In notebook 3, RT-VLM (captioning)
adds a fourth name: a file id that names the caption index.

<img src="../images/course/three-names.png" width="960" alt="One recording, traffic_short.mp4, with its four names wired to where each is used. The file stem traffic_short filters RT-CV detections and Behavior Analytics tracklet summaries, the VIOS name traffic_short_1 is for display and lookup only, the VIOS sensorId cuts clips, and the RT-VLM file id names the caption index default_&lt;file id&gt;.">

`course_setup.find_sensor()` looks a video up in VIOS by its file stem. Notebooks 3 and 4 use it to find this clip
again.

In [ ]:
vios = find_sensor(SOURCE)
if vios is None:
    raise RuntimeError(f"VIOS lists no source named {SOURCE} or {SOURCE}_<n>; run the upload cells above")
VIOS_NAME = vios["name"]
print("file stem      :", SOURCE)
print("VIOS name      :", VIOS_NAME)
print("VIOS sensor id :", vios["sensorId"],
      "(the copy this notebook uses)" if vios["sensorId"] == SENSOR_ID else "(another copy of the same file)")

Now ask VIOS for the recording's timeline: the time range it can cut clips from.

In [ ]:
r = requests.get(BASE + "/vios/vst/api/v1/storage/" + SENSOR_ID + "/timelines", timeout=20)
r.raise_for_status()
r.json()

The timeline covers about 16 seconds. It starts at the time you gave in the upload, which is a course choice, not
the recording date.

RT-CV's records of an uploaded file start at 2025-01-01T00:00:00, whatever start time the upload gives, and are
stored in `mdx-raw-2025-01-01`. Your clip's VIOS timeline starts at the same moment, so one start time, `ANCHOR`,
works for VIOS and for Elasticsearch. A live camera uses the real clock instead (section 2.13).

## 2.4 Watch the clip

Watch the video before you look at any model output. **Predict before you play it.** The camera looks at a road
intersection for 16 seconds. What event would a traffic operator want to be told about?

`course_setup.clip_bytes()` asks VIOS to cut a time range and downloads the clip. `labtools.show_video()` plays
it.

In [ ]:
video = clip_bytes(SENSOR_ID, ANCHOR, iso_offset(ANCHOR, 16))
labtools.show_video(video)

**Check your prediction.** Two vehicles, a white pickup and a red sedan, collide at about 3.5 to 4 seconds. They
stay in contact until about 10 seconds, pull apart at about 10.5 to 11 seconds, and stand apart from about 14
seconds.

Keep these times in mind. You use them again in section 2.11.

## 2.5 Wait for RT-CV's detections

RT-CV publishes its detections to Kafka, and Logstash indexes them in Elasticsearch. The first detection records
usually land within a minute.

**Predict before you run.** RT-CV writes one record per processed frame. How many records will a 16-second clip
produce?

The next cell counts the clip's records every 10 seconds until the count is above zero, and gives up after 20
minutes. A nonzero count means records are arriving, not that every frame is done.

In [ ]:
# Counts this clip's RT-CV records every 10 s; raises after 20 minutes.
frames = wait_for_detections(INDEX, SOURCE)
print(frames, "frame records so far; more may still be on the way")

## 2.6 Read the detections from Kafka

Kafka carries results between services. Producers write messages to named topics, and consumers read the topics
they need. RT-CV writes one message per frame to `mdx-raw`, and Logstash and Behavior Analytics both read it.

Start with the topic offsets. Each message adds one to its partition's offset. The cell adds up each
topic's offsets, so the `mdx-raw` total counts every frame message RT-CV has written so far, for all videos. It is
not a count of objects.

In [ ]:
import subprocess

proc = subprocess.run(
    ["docker", "exec", "kafka", "kafka-get-offsets",
     "--bootstrap-server", "localhost:9092"],
    capture_output=True, text=True, timeout=120,
)
if proc.returncode:
    raise RuntimeError("kafka-get-offsets failed: " + proc.stderr.strip()[-300:])

totals = {}
for line in proc.stdout.splitlines():
    topic, _, count_ = line.rsplit(":", 2)
    if topic.startswith("mdx-"):
        totals[topic] = totals.get(topic, 0) + int(count_)

for topic, n in sorted(totals.items()):
    if n:
        print("%-28s %d" % (topic, n))

The topics this course uses:

| Topic | Contents |
|---|---|
| `mdx-raw` | RT-CV frame and object metadata |
| `mdx-behavior` | Behavior Analytics tracklet summaries |
| `mdx-vlm-captions` | RT-VLM captions |

You request captions in notebook 3, so that topic may not appear yet. Other `mdx-` topics in the list belong to
services that later notebooks use. If a topic you expect is missing,
check the service that writes it.

### Read one detection message

A Kafka topic is split into partitions. RT-CV uses each message's source name as its key. Kafka sends all messages
with one key to one partition, so one partition holds your clip. The next cell reads the newest message of each
`mdx-raw` partition and keeps the one from `traffic_short`. The payload is binary Protobuf. The cell keeps it as
bytes and turns off the consumer's line separator, so no newline is added.

In [ ]:
import sys
if "/opt/vss/tools/message-broker-consumers" not in sys.path:
    sys.path.insert(0, "/opt/vss/tools/message-broker-consumers")
from schema_pb2 import Frame   # the message type, explained below

# RT-CV uses the source name as the message key, so every frame of one clip sits in one partition.
# Read the newest message of each mdx-raw partition and keep the one from this clip.
ends = {}
for line in proc.stdout.splitlines():   # proc: the kafka-get-offsets run above
    topic, partition, end = line.rsplit(":", 2)
    if topic == "mdx-raw" and int(end) > 0:
        ends[int(partition)] = int(end)

payload = None
for partition, end in sorted(ends.items()):
    result = subprocess.run(
        ["docker", "exec", "kafka", "timeout", "30",
         "kafka-console-consumer", "--bootstrap-server", "localhost:9092",
         "--topic", "mdx-raw", "--partition", str(partition), "--offset", str(end - 1),
         "--max-messages", "1", "--formatter-property", "line.separator="],
        capture_output=True, timeout=120,
    )
    if result.returncode or not result.stdout:
        print(f"partition {partition}: nothing read")
        continue
    newest = Frame()
    newest.ParseFromString(result.stdout)
    print(f"partition {partition}: newest message from {newest.sensorId}")
    if newest.sensorId == SOURCE:
        payload = result.stdout
        break

if payload is None:
    raise RuntimeError(f"no mdx-raw partition ends with a message from {SOURCE}: another video was processed after it, "
                       "or its messages expired (Kafka keeps them for a few hours). Check section 2.5, then rerun "
                       "the offsets cell and this one.")
print(len(payload), "bytes received; decode them in the next cell.")

### Decode the Protobuf payload

RT-CV publishes `nv.Frame` messages. The VSS source in the lab, `/opt/vss`, holds their definition,
`libs/nvschema/src/main/protobuf/schema.proto`, and the Python module generated from it,
`tools/message-broker-consumers/schema_pb2.py`. Always decode with the schema from the producer's VSS release.

`ParseFromString` turns the bytes into a typed `Frame`. The cell prints at most five objects and the first eight
values of each embedding.

In [ ]:
import sys
import json
from pathlib import Path

schema_dir = Path("/opt/vss/tools/message-broker-consumers")
if not (schema_dir / "schema_pb2.py").is_file():
    raise FileNotFoundError("VSS schema module is missing; check deployment in notebook 1.")
if str(schema_dir) not in sys.path:
    sys.path.insert(0, str(schema_dir))
from schema_pb2 import Frame

frame = Frame()
frame.ParseFromString(payload)
print("Frame:", frame.id, "Source:", frame.sensorId)
print("Timestamp:", frame.timestamp.ToJsonString())
print("Detected objects:", len(frame.objects), "(showing up to 5)")

for obj in frame.objects[:5]:
    summary = {
        "tracklet_id": obj.id, "type": obj.type,
        "confidence": round(obj.confidence, 4),
        "bbox": {key: round(getattr(obj.bbox, key), 1)
                 for key in ("leftX", "topY", "rightX", "bottomY")},
        "embedding_dimensions": len(obj.embedding.vector),
        "embedding_preview": [round(x, 4) for x in obj.embedding.vector[:8]],
    }
    print(json.dumps(summary, indent=2))


Look at `Source`: it is the file stem, `traffic_short`, the same name RT-CV writes to Elasticsearch. A frame with
no detections is still a valid message.

In your own service, pass each Kafka record's value bytes to `Frame.ParseFromString`. Keep message boundaries
intact: never split the bytes on newlines. Other topics carry other message types, so do not decode them all as a
`Frame`.

## 2.7 Read the same detections from Elasticsearch

Logstash indexes the `mdx-raw` messages of uploaded files in `mdx-raw-2025-01-01`, one record per frame. The index
holds every uploaded file, including the four search videos. To read only your clip, filter RT-CV's
`sensorId.keyword` field on the file stem, `traffic_short`.

**Your turn:** Write `source_filter(name)`, an Elasticsearch `bool` query that matches a record when
`sensorId.keyword` equals `name` **or** starts with `name` followed by `_`. The second form also catches a file
uploaded under a numbered name, such as `traffic_short_2.mp4`: RT-CV names records after the filename you send to
`/complete`, never after the VIOS name. It mirrors the rule `find_sensor()` uses for VIOS names. Replace each
`FIXME` and run the check cell. It prints `source_filter works` when your query is right. Click the `...` under the
cell for the solution.

In [ ]:
# TODO
def source_filter(name):
    """Query matching RT-CV records of one source: the exact file stem, or the
    stem followed by "_" and a number."""
    return {"bool": {
        "should": [
            {"term":   {FIXME: name}},
            {"prefix": {"sensorId.keyword": FIXME}},
        ],
        "minimum_should_match": 1,
    }}

In [ ]:
# SOLUTION
def source_filter(name):
    """Query matching RT-CV records of one source: the exact file stem, or the
    stem followed by "_" and a number."""
    return {"bool": {
        "should": [
            {"term":   {"sensorId.keyword": name}},
            {"prefix": {"sensorId.keyword": name + "_"}},
        ],
        "minimum_should_match": 1,
    }}

In [ ]:
query = source_filter(SOURCE)
if '"sensorId.keyword"' not in json.dumps(query) or SOURCE + "_" not in json.dumps(query):
    raise ValueError("source_filter must query sensorId.keyword for the exact name and for the name + '_' prefix")
from course_setup import source_filter as reference_filter   # the same query, as course_setup ships it
if query != reference_filter(SOURCE):
    raise ValueError("your query differs from course_setup.source_filter(SOURCE); compare it with the solution")

frames = es("POST", "/mdx-raw-2025-01-01/_count", json={"query": query})["count"]
stray = es("POST", "/mdx-raw-2025-01-01/_count", json={"query": source_filter("no_such_source")})["count"]
if frames == 0:
    raise ValueError(f"the filter selects no records for {SOURCE}; compare it with the solution")
if stray:
    raise ValueError("the filter also matches a source that does not exist; it is too wide")
print(f"source_filter works: {frames} frame records for {SOURCE}, 0 for a source that does not exist")

### Read one detection record

Fetch the earliest record for your clip that has objects in it. It describes one frame: each object's label, box,
confidence and tracklet id, plus an object embedding for each object.

In [ ]:
hits = es("POST", "/mdx-raw-2025-01-01/_search",
          json={"size": 5, "sort": [{"timestamp": "asc"}], "query": source_filter(SOURCE)})["hits"]["hits"]
if not hits:
    raise RuntimeError(f"no detection record for {SOURCE}; re-run the wait cell in section 2.5")
# The first frames of a clip can be empty while vehicles enter: keep the earliest record that has objects.
with_objects = [h["_source"] for h in hits if h["_source"].get("objects")]
if not with_objects:
    raise RuntimeError(f"the five earliest frames for {SOURCE} hold no objects; wait for more records, then re-run")
doc = with_objects[0]
print("frame at:", doc["timestamp"])
print("objects :", len(doc["objects"]), "\n")

obj = doc["objects"][0]
# Select readable fields explicitly; nested metadata can also hold vectors.
readable = {key: obj.get(key) for key in ("id", "type", "confidence")}
readable["bbox"] = {key: obj.get("bbox", {}).get(key)
                    for key in ("leftX", "topY", "rightX", "bottomY")}
print(json.dumps(readable, indent=2))

print("\nthis object also carries an embedding of",
      len(obj.get("embedding", {}).get("vector", [])), "floats")

These fields come straight from RT-CV, so your application can count, filter and draw them without another model
call.

The cell does not print the object embedding, which has 1,152 values. RT-CV's SigLIP v2 image encoder computes
one for each detected object (very small objects can have none), and objects that look alike get similar vectors. Section
2.12 searches with them, and so does the Part 1 assessment.

Now put every frame record of the clip into a table, one row per detected object. The request leaves out the
embeddings, which the table does not need.

In [ ]:
r = requests.post(BASE + "/es/" + INDEX + "/_search", timeout=60, json={
    "query": {"term": {"sensorId.keyword": SOURCE}},
    "size": 2000,
    "sort": [{"timestamp": {"order": "asc"}}],
    "_source": {"excludes": ["objects.embedding"]},   # boxes and ids only
})
r.raise_for_status()
df = labtools.objects_table(r.json()["hits"]["hits"])
if df.empty:
    raise RuntimeError(f"no detections returned for {SOURCE}")
print(len(df), "detections across", df["time"].nunique(), "frames")
df.head(10)

Each row is one object in one frame. The same id on many rows follows one object through the clip: that run of
detections is a tracklet, and its id is the tracklet id. Before you count vehicles from tracklet ids, look for
broken tracklets and gaps. A broken tracklet is one vehicle that gets a new id. A gap is a run of frames where the
detector missed the vehicle. A `confidence` of -0.1 marks a box that the tracker carried over on a frame where the
detector did not run.

## 2.8 Choose Kafka or Elasticsearch

You have now read the same detections both ways:

| | Kafka | Elasticsearch |
|---|---|---|
| Access pattern | Consume messages as they arrive | Query stored records |
| Detection format here | Protobuf | JSON |
| Example use | Act when a new detection arrives | Search, aggregate or fill a dashboard |
| How long data stays | A few hours on this deployment | Until the index lifecycle policy deletes the index |

A service that reacts to new detections subscribes to Kafka. A tool that looks back over a time range queries
Elasticsearch. One application can use both, and Logstash is itself a Kafka consumer.

### List the indices

List the indices the pipeline has created. If Kafka has messages but the index you expect is empty, check the
path in between: Logstash and Elasticsearch.

In [ ]:
r = requests.get(BASE + "/es/_cat/indices/mdx-*,default_*?v", timeout=20)
r.raise_for_status()
print(r.text)

The indices for uploaded files end in `2025-01-01`, as section 2.3 explained.

Find the `docs.count` column for `mdx-raw-2025-01-01`. **Predict before you run.** Is `docs.count` the number of
frame records? The next cell counts your clip's frame records with `_count` and your `source_filter`. Compare the
result with your prediction from section 2.5 too.

In [ ]:
frames = es("POST", "/mdx-raw-2025-01-01/_count", json={"query": source_filter(SOURCE)})["count"]
print("frames:", frames)

**Check your prediction.** `docs.count` is far larger than the number of frames, and not only because the index
holds other clips. `objects` is a nested field, so Elasticsearch stores each detected object as a hidden document
of its own, and `docs.count` counts those too. To count frames, use `_count`. For your clip, `_count`
should be close to the number of frames in the video, about 385: the clip runs a little over 16 seconds at 24
frames per second.

<img src="../images/course/nested-docs.png" width="960" alt="Three frame records of mdx-raw-2025-01-01, each a root document holding its detected objects as nested documents. On one measured deployment, _count counts the root documents only (385 frames), while docs.count counts roots and nested objects together (385 + 2,832 = 3,217).">

## 2.9 Draw detections on a frame

Cut one second of footage, decode a frame and draw the boxes on it. It is the quickest way to check RT-CV's boxes
and labels.

`iso(seconds)` returns `ANCHOR` plus that many seconds. As section 2.3 showed, the same timestamp works in VIOS and
in Elasticsearch.

In [ ]:
def iso(seconds):
    """Offset into the clip, in seconds -> timestamp on the shared 2025-01-01 timeline."""
    return iso_offset(ANCHOR, seconds)


SECONDS = 6
picture = labtools.frame_from_mp4(clip_bytes(SENSOR_ID, iso(SECONDS), iso(SECONDS + 1)))   # one decoded frame
print(picture.size)
picture

**Your turn:** Complete the query so it returns the first detection record at or after `SECONDS` for this clip.
Replace each `FIXME`. The first one asks which name RT-CV writes to `sensorId.keyword`: `SOURCE`, `VIOS_NAME` or
`SENSOR_ID`. The check cell reads the query you sent, then draws the boxes. Click the `...` under the cell for the
solution.

In [ ]:
# TODO
r = requests.post(BASE + "/es/" + INDEX + "/_search", timeout=30, json={
    "size": 1,
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": FIXME}},            # which identifier names the source here: SOURCE, VIOS_NAME or SENSOR_ID?
        {"range": {"timestamp": {"gte": FIXME}}},         # at or after SECONDS, as an indexed timestamp
    ]}},
    "sort": [{"timestamp": {"order": FIXME}}],            # earliest first, or latest first?
})
r.raise_for_status()
hits = r.json()["hits"]["hits"]
if not hits:
    raise RuntimeError(f"no frame record at or after {iso(SECONDS)}; a wrong identifier in the term filter returns nothing and no error")
objects = hits[0]["_source"]["objects"]
print(len(objects), "objects in the first frame at or after", iso(SECONDS))

In [ ]:
# SOLUTION
r = requests.post(BASE + "/es/" + INDEX + "/_search", timeout=30, json={
    "size": 1,
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": SOURCE}},           # the file stem: not the VIOS name, not the VIOS UUID
        {"range": {"timestamp": {"gte": iso(SECONDS)}}},  # the offset converted with the file anchor
    ]}},
    "sort": [{"timestamp": {"order": "asc"}}],            # earliest first = the first frame at or after SECONDS
})
r.raise_for_status()
hits = r.json()["hits"]["hits"]
if not hits:
    raise RuntimeError(f"no frame record at or after {iso(SECONDS)}; a wrong identifier in the term filter returns nothing and no error")
objects = hits[0]["_source"]["objects"]
print(len(objects), "objects in the first frame at or after", iso(SECONDS))

In [ ]:
# The check reads the query you sent: the file stem in the term filter, iso(SECONDS) as the start, earliest first.
sent = json.loads(r.request.body)
term, window = sent["query"]["bool"]["filter"]
used = term["term"]["sensorId.keyword"]
if used != SOURCE:
    raise ValueError(f"the term filter used {used!r}; RT-CV writes the file stem {SOURCE!r} to sensorId.keyword, "
                     f"not the VIOS name {VIOS_NAME!r} and not the VIOS sensor id {SENSOR_ID!r}")
start = window["range"]["timestamp"]["gte"]
if not isinstance(start, str) or offset_seconds(ANCHOR, start) != SECONDS:
    raise ValueError(f"the range starts at {start!r}; start it at iso(SECONDS), which is {iso(SECONDS)!r}")
if sent["sort"][0]["timestamp"]["order"] != "asc":
    raise ValueError("sort ascending: the first record at or after SECONDS is the earliest one")
if not objects:
    raise RuntimeError("that frame record lists no objects; change SECONDS by one and rerun from the clip cell")
labtools.draw_boxes(picture, objects)

Each label shows the class and the tracklet id. Compare a box with the vehicle under it. A box that lags or leads the
vehicle is a localization error. A wrong word is a label error. Boxes are in the video's own pixel coordinates, so
scale them if you resize a frame.

**Optional:** set `SECONDS` to 2 or 12, then rerun the clip cell, your query and the check. Does the same vehicle
keep its tracklet id?

## 2.10 Group detections into tracklets

A vehicle in view gives one detection per frame, all with the same tracklet id. The table in section 2.7 printed the
number of detections and frames. **Predict before you run.** How many distinct tracklets do those detections form?

`objects` is a nested field, so the query counts inside a `nested` aggregation on the `objects` path.

In [ ]:
r = requests.post(BASE + "/es/" + INDEX + "/_search", timeout=30, json={
    "query": {"term": {"sensorId.keyword": SOURCE}},
    "size": 0,
    "aggs": {"objs": {
        "nested": {"path": "objects"},
        "aggs": {
            "by_type": {"terms": {"field": "objects.type.keyword", "size": 20}},
            "tracklets": {"cardinality": {"field": "objects.id.keyword"}},
        }}},
})
r.raise_for_status()
agg = r.json()["aggregations"]["objs"]
buckets = agg["by_type"]["buckets"]
tracklets = agg["tracklets"]["value"]
if not buckets:
    raise RuntimeError(f"the aggregation returned no classes for {SOURCE}")

for b in buckets:
    print("%-12s %d" % (b["key"], b["doc_count"]))

total = sum(b["doc_count"] for b in buckets)
print("\ntotal detections:", total)
print("distinct tracklet ids:", tracklets)
print("about %d detections per tracklet" % (total / max(tracklets, 1)))

In [ ]:
labtools.plot_counts([b["key"] for b in buckets],
                     [b["doc_count"] for b in buckets],
                     title="What RT-CV saw", xlabel="detections")

**Check your prediction.** There are far fewer tracklets than detections: one tracked vehicle can account for
hundreds of detections. The exact counts change a little from run to run.

<img src="../images/course/boxes-to-tracklet.gif" width="960" alt="Seven frame records of the clip hold 14 detections. Each car's detections link into its own tracklet, tracklet A or tracklet B. Each tracklet's summary card holds the seconds RT-CV followed it and how far its box moved after 5 seconds. A rule on those two numbers keeps the cars that stand still and drops the cars that move. A closing note reads: A rule on movement finds cars that stand still. It cannot tell a parked car from a crash.">

The number of distinct tracklet ids counts tracklets, not vehicles. A broken tracklet counts one vehicle twice, and
an id switch can merge two vehicles into one tracklet.

**Your turn:** Follow one vehicle. Pick a tracklet id you can read on the overlay from section 2.9. Replace the `FIXME`
with that id and run the cell. It reports for how many frames RT-CV followed the tracklet and how far its box moved. The
check cell asks Elasticsearch how many frames hold that tracklet. That number should match the rows your cell found. Click
the `...` under the cell for the solution.

In [ ]:
# TODO
TRACKLET = FIXME                   # a tracklet id printed on the overlay
TRACKLET = str(TRACKLET)           # tracklet ids are strings in the records

one = df[df["tracklet"].astype(str) == TRACKLET].sort_values("time")
if one.empty:
    raise ValueError(f"tracklet {TRACKLET!r} is not in the table; copy an id from the overlay labels")
first, last = one.iloc[0], one.iloc[-1]
moved_px = abs((last["left"] + last["right"]) / 2 - (first["left"] + first["right"]) / 2)
print(f"tracklet {TRACKLET} ({first['type']}): {len(one)} frames, "
      f"{str(first['time'])[11:23]} to {str(last['time'])[11:23]}, "
      f"box center moved {moved_px:.0f} px horizontally")

In [ ]:
# SOLUTION
# Any id you can read on the overlay is a correct answer. This picks one without
# knowing the ids in advance: the largest box in the frame, usually the nearest vehicle.
def area(o):
    b = o.get("bbox", {}) or {}
    return max(0, b.get("rightX", 0) - b.get("leftX", 0)) * max(0, b.get("bottomY", 0) - b.get("topY", 0))


TRACKLET = str(max(objects, key=area)["id"])

one = df[df["tracklet"].astype(str) == TRACKLET].sort_values("time")
if one.empty:
    raise ValueError(f"tracklet {TRACKLET!r} is not in the table; copy an id from the overlay labels")
first, last = one.iloc[0], one.iloc[-1]
moved_px = abs((last["left"] + last["right"]) / 2 - (first["left"] + first["right"]) / 2)
print(f"tracklet {TRACKLET} ({first['type']}): {len(one)} frames, "
      f"{str(first['time'])[11:23]} to {str(last['time'])[11:23]}, "
      f"box center moved {moved_px:.0f} px horizontally")

In [ ]:
# Ask the index how many frames of this clip contain the tracklet.
r = requests.post(BASE + "/es/" + INDEX + "/_count", timeout=30, json={
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": SOURCE}},
        {"nested": {"path": "objects", "query": {"term": {"objects.id.keyword": TRACKLET}}}},
    ]}},
})
r.raise_for_status()
indexed = r.json()["count"]
if indexed == 0:
    raise RuntimeError(f"the index holds no frame with tracklet {TRACKLET} for {SOURCE}")
print(f"frames in the index with tracklet {TRACKLET}: {indexed}; rows in your table: {len(one)}")
if indexed > len(one):
    print("RT-CV wrote more records after you built the table: rerun the table cell in section 2.7")

## 2.11 Find the vehicles that stand still

A vehicle that stands still keeps its box in the same place, frame after frame. Two places can show that:

- Behavior Analytics, a VSS service that summarizes RT-CV's tracklets. Each tracklet summary holds `speed`,
  `distance` and `timeInterval`, with `timestamp` and `end` marking the interval.
- RT-CV's own tracklets: the table from section 2.7.

First see what Behavior Analytics wrote for this clip, next to the cars RT-CV tracked.

In [ ]:
r = requests.post(BASE + "/es/mdx-behavior-*/_search", timeout=30, json={
    "query": {"term": {"sensor.id.keyword": SOURCE}},
    "size": 10000,    # Behavior Analytics can write about one record per frame
    "_source": ["object", "speed", "distance", "timeInterval", "timestamp", "end"],
})
r.raise_for_status()
summaries = [h["_source"] for h in r.json()["hits"]["hits"]]
if not summaries:
    raise RuntimeError(f"no tracklet summaries for {SOURCE} in mdx-behavior-*: Behavior Analytics writes its records a "
                       "little after RT-CV's; wait a minute and rerun this cell, or check vss-behavior-analytics "
                       "in notebook 1")

cars = sorted(df.loc[df["type"] == "car", "tracklet"].astype(str).unique(), key=int)
print(f"RT-CV followed {len(cars)} car tracklets: {', '.join(cars)}")

records = {}                          # tracklet id -> how many summaries Behavior Analytics wrote for it
for s in summaries:
    tracklet = str(s["object"]["id"])
    records[tracklet] = records.get(tracklet, 0) + 1
print(f"Behavior Analytics wrote {len(summaries)} summaries for {len(records)} tracklets: "
      + ", ".join(f"tracklet {t} ({records[t]})" for t in sorted(records, key=int)))

longer = sorted((s for s in summaries if s["timeInterval"] > 0), key=lambda s: s["timeInterval"], reverse=True)
print(f"\n{len(longer)} of the {len(summaries)} summaries cover more than one frame (timeInterval above 0 s), "
      "longest first:")
for s in longer:
    print(f"tracklet {str(s['object']['id']):<5} speed {s['speed']:6.2f}  distance {s['distance']:7.2f}  "
          f"{s['timeInterval']:6.2f} s  {str(s['timestamp'])[11:23]} to {str(s['end'])[11:23]}")

Compare the two lists. How many of RT-CV's car tracklets does Behavior Analytics summarize, and for how long? The
answer changes from run to run. In one test run (2026-11-03) it summarized 3 of 7 car tracklets, mostly for a
single frame. In another (2026-11-05) it summarized every car over its whole time in view. A summary that can skip
cars cannot be the only place you look for the ones that stand still. Check what a service writes before you
build a rule on it.

For vehicles that stand still, use RT-CV's own tracklets: they hold every car on every run.

For each car tracklet, the next cell prints how long RT-CV followed it. It also prints how far the middle of its
box moved after the first 5 seconds. By then, the cars in view have driven in. A car that stays in place, such
as a parked car, barely moves. This clip's traffic crosses the frame, so the side-to-side movement is enough.

In [ ]:
import pandas as pd

# One row per car tracklet: how long RT-CV followed it, and how far its box moved after the first 5 seconds.
SETTLE_S = 5    # ignore the first 5 seconds, while cars drive in

rows = []
for tracklet, group in df[df["type"] == "car"].groupby("tracklet"):
    at = group["time"].map(lambda t: offset_seconds(ANCHOR, t))
    center = (group["left"] + group["right"]) / 2      # side-to-side position of the box, in pixels
    late = center[at >= SETTLE_S]
    rows.append({"tracklet": str(tracklet), "start_s": at.min(), "seconds": at.max() - at.min(),
                 "moved_after_px": late.max() - late.min() if len(late) else float("nan")})
if not rows:
    raise RuntimeError(f"the table from section 2.7 holds no car for {SOURCE}; rerun the table cell in section 2.7")
car_tracklets = pd.DataFrame(rows).sort_values("seconds", ascending=False)
for t in car_tracklets.itertuples():
    moved = "gone before 5 s" if pd.isna(t.moved_after_px) else f"moved {t.moved_after_px:4.0f} px after 5 s"
    print(f"tracklet {t.tracklet:<5} followed {t.seconds:5.1f} s (from {t.start_s:4.1f} s)  {moved}")

**Your turn:** Turn the list into a shortlist of cars that stand still. Replace each `FIXME`. `MIN_SECONDS` is
the shortest time RT-CV must follow a car. `MAX_MOVE_PX` is the most its box may move after 5 seconds. A good
rule keeps the cars that stand still and drops the cars that move. **Predict before you run.** How many cars
will pass your rule? Click the `...` under the cell for the solution.

In [ ]:
# TODO
MIN_SECONDS = FIXME    # keep cars that RT-CV followed for at least this many seconds
MAX_MOVE_PX = FIXME    # and whose box moved at most this many pixels after the first 5 seconds

shortlist = car_tracklets[(car_tracklets["seconds"] >= MIN_SECONDS) & (car_tracklets["moved_after_px"] <= MAX_MOVE_PX)]
print(f"{len(shortlist)} of {len(car_tracklets)} car tracklets qualify: followed at least {MIN_SECONDS} s, "
      f"moved at most {MAX_MOVE_PX} px after 5 s")

In [ ]:
# SOLUTION
# A rule to start from. Yours may differ: cars that stand still should pass,
# and cars that move should not.
MIN_SECONDS = 8
MAX_MOVE_PX = 25

shortlist = car_tracklets[(car_tracklets["seconds"] >= MIN_SECONDS) & (car_tracklets["moved_after_px"] <= MAX_MOVE_PX)]
print(f"{len(shortlist)} of {len(car_tracklets)} car tracklets qualify: followed at least {MIN_SECONDS} s, "
      f"moved at most {MAX_MOVE_PX} px after 5 s")

In [ ]:
if shortlist.empty:
    raise RuntimeError("no car passes your rule; raise MAX_MOVE_PX or lower MIN_SECONDS in your # TODO cell and "
                       "rerun it. The cars that stand still should pass.")
if len(shortlist) == car_tracklets["moved_after_px"].notna().sum():    # the car tracklets RT-CV saw after 5 s
    raise RuntimeError("Your rule keeps every car RT-CV saw after 5 seconds, so it filters nothing: lower "
                       "MAX_MOVE_PX or raise MIN_SECONDS in your # TODO cell and rerun it.")
for t in shortlist.itertuples():
    print(f"tracklet {t.tracklet:<5} followed {t.seconds:5.1f} s (from {t.start_s:4.1f} s)  "
          f"moved {t.moved_after_px:4.0f} px after 5 s")
print(f"\n{len(shortlist)} of {len(car_tracklets)} car tracklets pass your rule")

Compare the count with your prediction. Then match the shortlist to the clip from section 2.4. The overlay from
section 2.9 labels each box with its tracklet id. A good shortlist holds cars that stood still the whole time, such
as parked cars.

Where are the two cars from the collision? In the list of all car tracklets above, they are the two that
moved most after 5 seconds. They stopped at about 4 seconds and pulled apart at about 10.5 seconds. So a rule for cars
that stand still drops them. Is one of them on your shortlist? Then your rule lets moving cars through: lower
`MAX_MOVE_PX`, then rerun your `# TODO` cell and the check cell.

Movement data shows where cars are and how they move, but not that two of them collided. Notebook 3 asks a VLM
what happened.

## 2.12 Search for objects by appearance

RT-CV computes an object embedding for each object it detects (section 2.7). Objects that look alike have similar
embeddings, so a nearest-neighbor (kNN) search over the object vectors finds objects by appearance. You can ask in two ways:

- **By example:** start from an object RT-CV detected and find objects that look like it. The VSS user interface
  calls this search by image. RT-CV has no route that embeds a photo you upload.
- **By text:** RT-CV's text encoder turns a phrase such as "a white car" into a vector in the same space.

Both search `objects.embedding.vector` in `mdx-raw`. Either way you find objects, not events.

The next cell defines the search. It takes a vector and returns the five frames whose objects come closest
to it, with the object that matched in each frame.

In [ ]:
def similar_objects(vector, k=5):
    """The k frames of this clip whose best object is closest to `vector`, each with that object."""
    body = {
        "size": k,
        "_source": {"excludes": ["objects.embedding"]},          # boxes and ids, not 1,152 numbers per object
        "query": {"bool": {
            "filter": [{"term": {"sensorId.keyword": SOURCE}}],
            "must": [{"nested": {
                "path": "objects", "score_mode": "max",
                "query": {"knn": {"field": "objects.embedding.vector", "query_vector": vector,
                                  "k": k, "num_candidates": 100,
                                  "filter": {"term": {"sensorId.keyword": SOURCE}}}},
                "inner_hits": {"size": 1, "_source": False},      # which object in the frame matched
            }}],
        }},
    }
    matches = []
    for hit in es("POST", "/" + INDEX + "/_search", json=body)["hits"]["hits"]:
        best = hit["inner_hits"]["objects"]["hits"]["hits"][0]
        obj = hit["_source"]["objects"][best["_nested"]["offset"]]
        matches.append({"time": hit["_source"]["timestamp"], "tracklet": str(obj.get("id")),
                        "type": str(obj.get("type")), "score": round(best["_score"], 3), "object": obj})
    return matches


def show_matches(matches):
    for m in matches:
        print(f"{str(m['time'])[11:23]}  tracklet {m['tracklet']:>5}  {m['type']:<10}  score {m['score']}")

**Search by example.** Use the vehicle you followed in section 2.10, as it appears in the frame from section 2.9.
The top hits should carry the same tracklet id: the same vehicle in other frames of the clip. They need not include
the frame from section 2.9, and several hits can share one score, because the vehicle looks almost the same from
frame to frame.

In [ ]:
example = next((o for o in objects if str(o.get("id")) == TRACKLET), None)
if example is None:
    raise ValueError(f"tracklet {TRACKLET} is not in the frame from section 2.9; set TRACKLET to an id on that overlay "
                     "and rerun section 2.10")
example_vector = example.get("embedding", {}).get("vector", [])
if not example_vector:
    raise RuntimeError(f"tracklet {TRACKLET} has no embedding in this frame; pick another vehicle in section 2.10")
print(f"example: tracklet {TRACKLET} ({example.get('type')}), {len(example_vector)} values\n")
show_matches(similar_objects(example_vector))

**Search by text.** Ask RT-CV's text encoder for the vector of a short phrase, then run the same search. The phrase
lands in the same space as the object vectors, so the closest objects should fit the words.

In [ ]:
TEXT = "a white car"

r = requests.post(BASE + "/rtvi-cv/api/v1/generate_text_embeddings",
                  json={"text_input": TEXT, "model": ""}, timeout=60)
r.raise_for_status()
item = r.json()["data"][0]
text_vector = item["embedding"] if isinstance(item, dict) else item   # the route answers in one of two shapes
print(f"{TEXT!r} -> {len(text_vector)} values\n")
text_matches = similar_objects(text_vector)
show_matches(text_matches)

In [ ]:
# Draw the best match on its frame to check it against the footage.
best = text_matches[0]
at = offset_seconds(ANCHOR, best["time"])
print(f"best match for {TEXT!r}: tracklet {best['tracklet']} ({best['type']}) at {at:.1f} s")
labtools.draw_boxes(labtools.frame_from_mp4(clip_bytes(SENSOR_ID, iso(at), iso(at + 1))), [best["object"]])

Try another phrase, such as "a red car" or "a person", and rerun the last two cells. A score is a similarity, not a
probability, so compare the ranking and check the top hits against the footage. A kNN search always returns its
nearest objects, even when none of them fits the words.

## 2.13 Optional: go further

Three extras for when you finish early. Nothing later depends on them, so skip to **Check your
understanding** if time is short.

### Plot trajectories

Plot where each tracklet went across the frame. The helper follows the bottom center of each box, roughly where the
vehicle meets the road.

In [ ]:
labtools.plot_paths(picture, df)

Look for paths that follow vehicles through the intersection. Sudden jumps can mean tracking errors. To stay
readable, the plot hides tracklets with little sideways movement, so a vehicle moving mainly toward the camera can be
missing. That is a display filter, not a test of whether an object moved.

### Browse the web interfaces

Open each page in a new browser tab: copy your JupyterLab address and replace everything after the host name
with the page's path.

- `/vios-ui/`: browse the videos in VIOS.
- `/kibana/`: explore the indexed records. In **Discover**, pick `mdx-raw-*` and a time range that includes
  2025-01-01.

### Register a live stream

A camera sends an RTSP stream, and its records carry the real clock time instead of 2025-01-01. NVStreamer loops a
file as an RTSP stream, so you can try the camera path without a camera. Nothing else in the course uses this
stream. Release it at the end, because it keeps the GPU busy while it runs.

**1. Loop the clip as an RTSP stream.** Upload the clip to NVStreamer with the same chunked upload as before, then
read the RTSP URL it returns.

In [ ]:
with open(CLIP, "rb") as fh:
    r = requests.post(
        BASE + "/nvstreamer/api/v1/storage/file",
        headers={
            "nvstreamer-chunk-number":  "1",
            "nvstreamer-total-chunks":  "1",
            "nvstreamer-is-last-chunk": "true",
            "nvstreamer-identifier":    "live_traffic.mp4",
            "nvstreamer-file-name":     "live_traffic.mp4",
        },
        files={
            "metadata": (None, json.dumps({"timestamp": "2025-01-01T00:00:00"})),
            "file": ("live_traffic.mp4", fh, "video/mp4"),
        },
        timeout=300,
    )
r.raise_for_status()
uploaded = r.json()
# An NVStreamer file upload can answer with an empty sensorId; streamId then names the file.
nvstreamer_id = uploaded.get("sensorId") or uploaded["streamId"]
remember("nvstreamer", nvstreamer_id, name="live_traffic")

started = time.time()
while True:
    r = requests.get(BASE + f"/nvstreamer/api/v1/sensor/{nvstreamer_id}/streams", timeout=30)
    r.raise_for_status()
    streams = [s for s in r.json() if s.get("streamId") and s.get("url")]
    elapsed = int(time.time() - started)
    if streams:
        break
    if elapsed > 30:
        raise TimeoutError(f"NVStreamer has not exposed the uploaded stream after {elapsed} s; check the upload response above")
    print(f"{elapsed:>4}s  waiting for NVStreamer to expose the stream ...")
    time.sleep(2)
remember("nvstreamer", nvstreamer_id, name="live_traffic", stream_id=streams[0]["streamId"])
rtsp = streams[0]["url"]
rtsp

The URL's path starts with `/nvstream/`.

**2. Register the stream and record it.** Register the URL with the VSS agent service as `Camera_1`. Any status
other than `success` stops the cell.

Registration does not switch on recording. The next cell finds `Camera_1` and switches on the VIOS recorder for its
main stream. It waits up to three minutes for the first recorded segment, which usually takes about a minute. The
cell after it waits for an `mdx-raw` index dated today, which shows that RT-CV is reading the stream.

In [ ]:
r = requests.post(BASE + "/agent/api/v1/rtsp-streams/add",
                  json={"sensorUrl": rtsp, "name": "Camera_1"}, timeout=120)
r.raise_for_status()
registration = r.json()
if registration.get("status") != "success":
    raise RuntimeError(f"Live stream registration failed: {registration}")
registration

In [ ]:
# Resolve this camera explicitly, not the first sensor in the deployment.
r = requests.get(BASE + "/vios/vst/api/v1/sensor/list", timeout=30)
r.raise_for_status()
cameras = [s for s in r.json() if s.get("name") == "Camera_1"]
if len(cameras) != 1:
    raise RuntimeError(f"Expected one Camera_1 source, found {len(cameras)}; inspect VIOS before continuing")
live_sensor_id = cameras[0]["sensorId"]
r = requests.get(BASE + f"/vios/vst/api/v1/sensor/{live_sensor_id}/streams", timeout=30)
r.raise_for_status()
main_streams = [s for s in r.json() if s.get("isMain")]
if len(main_streams) != 1:
    raise RuntimeError(f"Expected one main stream for Camera_1, found {len(main_streams)}")
live_stream_id = main_streams[0]["streamId"]
remember("live", live_sensor_id, name="Camera_1", stream_id=live_stream_id)

# The VSS agent service's registration API has no recording flag: switch the VIOS recorder on yourself.
ACTIVE = {"user", "alwaysOn"}
recorder = BASE + f"/vios/vst/api/v1/record/{live_stream_id}"
r = requests.get(recorder + "/status", timeout=30)
r.raise_for_status()
if r.json().get("recordingStatus") not in ACTIVE:
    requests.post(recorder + "/start", timeout=30).raise_for_status()
    print("recorder started")

started = time.time()
while True:
    r = requests.get(recorder + "/status", timeout=30)
    r.raise_for_status()
    status = r.json().get("recordingStatus")
    if status not in ACTIVE:
        raise RuntimeError(f"Recording is not active (recorder status {status!r}); check VIOS before requesting a clip")
    r = requests.get(BASE + f"/vios/vst/api/v1/storage/{live_stream_id}/timelines", timeout=30)
    r.raise_for_status()
    timelines = r.json()
    elapsed = int(time.time() - started)
    if timelines:
        print(f"{elapsed:>4}s  recorded timeline: {timelines}")
        break
    if elapsed > 180:
        raise TimeoutError(f"No recorded segment for Camera_1 after {elapsed} s; check the recorder status above and VIOS")
    print(f"{elapsed:>4}s  recorder {status}, waiting for the first recorded segment ...")
    time.sleep(10)

In [ ]:
# The recorder proves VIOS keeps the footage; this proves RT-CV is reading the stream.
from datetime import datetime, timezone

today = datetime.now(timezone.utc).date().isoformat()
started = time.time()
while True:
    r = requests.get(BASE + "/es/_cat/indices/mdx-raw-*?h=index", timeout=20)
    r.raise_for_status()
    live = [name for name in r.text.split() if name.endswith(today)]
    elapsed = int(time.time() - started)
    if live:
        print(f"{elapsed:>4}s  live-stream index present: {', '.join(live)}")
        break
    if elapsed > 180:
        raise TimeoutError(f"no mdx-raw index dated {today} after 180 s; check the registration above and RT-CV")
    print(f"{elapsed:>4}s  waiting for an mdx-raw index dated {today} ...")
    time.sleep(10)

r = requests.get(BASE + "/es/_cat/indices/mdx-*?v", timeout=20)
r.raise_for_status()
print(r.text)

Look for indices dated today. A query limited to the file's date never finds the stream's records, so check both
the source and the time range before you trust an empty result.

The first frames can land in a `1970-01-01` index, before the stream's timestamps are set. Do not trust the times
of those records.

**3. Remove the stream.** Remove the camera from the VSS agent service to stop its processing.

In [ ]:
r = requests.delete(BASE + "/agent/api/v1/rtsp-streams/delete/Camera_1", timeout=120)
r.raise_for_status()
r.json()

**4. Release what you started.** The recorded footage, the NVStreamer file and the stream's records are still
there. The next cell reads the live-stream entries (`live` and `nvstreamer`) from the list of what Part 1 created.
It shows what it will remove, removes it, and raises if anything is left. It never touches `traffic_short` or the four
search videos. Allow up to two minutes.

In [ ]:
from cleanup import Cleanup

release = Cleanup()
release.resources = [r for r in release.resources if r["kind"] in {"live", "nvstreamer"}]
release.preview()
release.run(confirm=True)

## Check your understanding

1. A colleague counts detections under `traffic_short_1`, the name VIOS shows, and gets zero. What is wrong?
2. You want an alarm the moment a new detection arrives, and a chart of yesterday's traffic. Which interface serves
   each?
3. Why does RT-CV report far more detections than tracklets? Is the number of tracklets the number of vehicles?
4. You need the red cars in the clip. Can you filter the class labels? What do you use instead?

<details>
<summary>Answers</summary>

1. RT-CV and Behavior Analytics records carry the file stem, `traffic_short`, as the source name. The VIOS name,
   with the suffix VIOS adds, is only for display and for `find_sensor()`.
2. Kafka for the alarm: it delivers messages as they arrive. Elasticsearch for the chart: it queries and
   aggregates stored records.
3. A vehicle is detected in every frame it is visible in, all under one tracklet id. Tracklets are not vehicles: a
   broken tracklet counts one vehicle twice, and an id switch can merge two.
4. No. Color is not a class. Embed the text "a red car" with RT-CV's text encoder and run a kNN search over the
   object vectors, as in section 2.12. Then check the top hits against the footage: kNN returns its nearest objects
   whether they are red or not.

</details>

## Next

Open [3. RT-VLM: Captioning & Prompting](03-rt-vlm-captioning-and-prompting.ipynb). RT-CV told you where the
vehicles are and how they move, but not that two of them collided. Notebook 3 asks a VLM to describe the scene. It
finds this clip again with `course_setup.find_sensor("traffic_short")`.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the check cell of the exercise in 2.11 prints how many cars pass your rule.
>
> **Then:** Break, 10 min.
>
> **Next: [3. RT-VLM: Captioning & Prompting](03-rt-vlm-captioning-and-prompting.ipynb)** &middot; Part 1 &middot; about 35 min  
> Caption the clip with RT-VLM and see how the chunk size, the sampled frames and the prompt change the answer.
>
> [Back to Home](../00_Home.ipynb)


<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>